# 01 Dataset Audit

Canonical structure- and label-audit for the MODMA/Lanzhou 2015 dataset. Full background (recording protocol, file-index → task-type mapping, xlsx schema, join recipe) is in [`DATA_CONTEXT.md`](DATA_CONTEXT.md) — this notebook verifies those facts against the data on disk and produces the labeled subject table every downstream notebook depends on.

Primary questions:
- Does every subject folder have exactly the expected 29 correctly-named recordings, with no missing/extra/corrupt/duplicate files?
- How many subjects are in each class (`MDD` / `HC`), and what do the demographics look like?
- Does every subject folder join 1:1 against the metadata workbook, with zero orphans on either side?
- What is the correct unit of patient-level leakage awareness for all later notebooks and modeling work?

Output: `backend/data/processed/subjects_labeled.csv` (52 rows, one per subject).


In [1]:
import sys
from pathlib import Path

_nb_dir = Path("backend/notebooks") if Path("backend/notebooks").exists() else Path.cwd()
if str(_nb_dir) not in sys.path:
    sys.path.insert(0, str(_nb_dir))

from _eda_utils import bootstrap_project_paths, save_processed

PROJECT_ROOT = bootstrap_project_paths()

from mendxai.core.config import config
from mendxai.ml.data_loader import DataLoader, classify_task, verify_data_structure

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"Dataset dir:  {config.data.dataset_dir}")
print(f"Metadata:     {config.data.metadata_path}")


PROJECT_ROOT: /Users/vishalkarda/Documents/Projects/mendx.ai
Dataset dir:  /Users/vishalkarda/Documents/Projects/mendx.ai/backend/data/audio_lanzhou_2015
Metadata:     /Users/vishalkarda/Documents/Projects/mendx.ai/backend/data/audio_lanzhou_2015/subjects_information_audio_lanzhou_2015.xlsx


In [2]:
verify_data_structure()



=== Data Structure Verification ===

Dataset directory exists: /Users/vishalkarda/Documents/Projects/mendx.ai/backend/data/audio_lanzhou_2015
  52 subject folders, 1508 .wav files
Metadata workbook exists: /Users/vishalkarda/Documents/Projects/mendx.ai/backend/data/audio_lanzhou_2015/subjects_information_audio_lanzhou_2015.xlsx

All data directories verified successfully!


True

In [3]:
import pandas as pd

EXPECTED_INDICES = set(range(1, 30))  # 01..29

subject_rows = []
naming_issues = []

for subject_dir in sorted(config.data.dataset_dir.iterdir()):
    if not subject_dir.is_dir():
        continue
    wav_files = sorted(subject_dir.glob("*.wav"))
    indices = set()
    bad_names = []
    for f in wav_files:
        try:
            indices.add(int(f.stem))
        except ValueError:
            bad_names.append(f.name)

    missing = sorted(EXPECTED_INDICES - indices)
    extra = sorted(indices - EXPECTED_INDICES)

    if missing or extra or bad_names:
        naming_issues.append({
            "subject": subject_dir.name, "missing": missing, "extra": extra, "bad_names": bad_names,
        })

    subject_rows.append({
        "subject_id": subject_dir.name,
        "n_wav_files": len(wav_files),
        "n_valid_indices": len(indices),
    })

subjects_df = pd.DataFrame(subject_rows).sort_values("subject_id").reset_index(drop=True)
display(subjects_df)
print(f"Total subjects: {len(subjects_df)}")
print(f"Total .wav files: {int(subjects_df['n_wav_files'].sum())}")

if naming_issues:
    print(f"\nNAMING ISSUES found in {len(naming_issues)} subject(s):")
    for issue in naming_issues:
        print(f"  {issue}")
else:
    print("\nNo naming issues: every subject has exactly files 01.wav..29.wav.")


,subject_id,n_wav_files,n_valid_indices
0,02010001,29,29
1,02010002,29,29
2,02010003,29,29
3,02010004,29,29
4,02010005,29,29
5,02010006,29,29
6,02010008,29,29
7,02010009,29,29
8,02010010,29,29
9,02010011,29,29


Total subjects: 52
Total .wav files: 1508

No naming issues: every subject has exactly files 01.wav..29.wav.


In [4]:
import hashlib
import soundfile as sf


def _file_fingerprint(path: Path) -> str:
    """Cheap content fingerprint (size + first/last 4KB hash) for duplicate
    detection without hashing all 1508 full audio files."""
    size = path.stat().st_size
    with open(path, "rb") as f:
        head = f.read(4096)
        f.seek(max(0, size - 4096))
        tail = f.read(4096)
    return hashlib.md5(head + tail + str(size).encode()).hexdigest()


unreadable = []
fingerprints = {}
duplicates = []

all_wavs = sorted(config.data.dataset_dir.glob("*/*.wav"))
for wav_path in all_wavs:
    try:
        sf.info(str(wav_path))  # header-only read; raises on corrupt/unreadable files
    except Exception as e:
        unreadable.append({"path": str(wav_path), "error": str(e)})
        continue

    fp = _file_fingerprint(wav_path)
    if fp in fingerprints:
        duplicates.append({"path": str(wav_path), "duplicate_of": fingerprints[fp]})
    else:
        fingerprints[fp] = str(wav_path)

print(f"Checked {len(all_wavs)} files")
print(f"Unreadable/corrupt files: {len(unreadable)}")
for u in unreadable:
    print(f"  {u}")
print(f"Exact duplicate files (by content fingerprint): {len(duplicates)}")
for d in duplicates:
    print(f"  {d}")


Checked 1508 files
Unreadable/corrupt files: 5
  {'path': '/Users/vishalkarda/Documents/Projects/mendx.ai/backend/data/audio_lanzhou_2015/02010004/24.wav', 'error': "Error opening '/Users/vishalkarda/Documents/Projects/mendx.ai/backend/data/audio_lanzhou_2015/02010004/24.wav': Format not recognised."}
  {'path': '/Users/vishalkarda/Documents/Projects/mendx.ai/backend/data/audio_lanzhou_2015/02010004/25.wav', 'error': "Error opening '/Users/vishalkarda/Documents/Projects/mendx.ai/backend/data/audio_lanzhou_2015/02010004/25.wav': Format not recognised."}
  {'path': '/Users/vishalkarda/Documents/Projects/mendx.ai/backend/data/audio_lanzhou_2015/02010004/26.wav', 'error': "Error opening '/Users/vishalkarda/Documents/Projects/mendx.ai/backend/data/audio_lanzhou_2015/02010004/26.wav': Format not recognised."}
  {'path': '/Users/vishalkarda/Documents/Projects/mendx.ai/backend/data/audio_lanzhou_2015/02010004/27.wav', 'error': "Error opening '/Users/vishalkarda/Documents/Projects/mendx.ai/back

In [5]:
metadata_df = DataLoader().load_metadata()
display(metadata_df.head())
print(f"\nShape: {metadata_df.shape}")
print(f"Dtypes:\n{metadata_df.dtypes}")



Loaded metadata for 52 subjects
Columns: ['subject id', 'type', 'age', 'gender', 'education（years）', 'PHQ-9', 'CTQ-SF', 'LES', 'SSRS', 'GAD-7', 'PSQI']


,subject id,type,age,gender,education（years）,PHQ-9,CTQ-SF,LES,SSRS,GAD-7,PSQI
0,2010002,MDD,18,F,12,23,77,-143,31,18,12
1,2010004,MDD,25,F,19,12,53,-44,38,13,11
2,2010005,MDD,20,M,16,19,49,-3,28,11,5
3,2010006,MDD,42,M,16,16,59,-30,40,12,9
4,2010008,MDD,42,M,12,17,66,-71,44,18,13



Shape: (52, 11)
Dtypes:
subject id          int64
type                  str
age                 int64
gender                str
education（years）    int64
PHQ-9               int64
CTQ-SF              int64
LES                 int64
SSRS                int64
GAD-7               int64
PSQI                int64
dtype: object


In [6]:
metadata_df = metadata_df.copy()
metadata_df["subject_id"] = metadata_df["subject id"].apply(lambda x: str(int(x)).zfill(8))

folder_ids = set(subjects_df["subject_id"])
xlsx_ids = set(metadata_df["subject_id"])

orphan_folders = sorted(folder_ids - xlsx_ids)
orphan_xlsx_rows = sorted(xlsx_ids - folder_ids)

print(f"Folders with no metadata match: {len(orphan_folders)} {orphan_folders}")
print(f"Metadata rows with no folder match: {len(orphan_xlsx_rows)} {orphan_xlsx_rows}")

assert not orphan_folders and not orphan_xlsx_rows, (
    "Subject-ID join is not 1:1 — see backend/notebooks/DATA_CONTEXT.md section 5 for the join recipe."
)
print("\nJoin verified: all 52 folders match exactly one metadata row, zero orphans either side.")

subjects_labeled_df = subjects_df.merge(metadata_df, on="subject_id", how="inner")
display(subjects_labeled_df.head())


Folders with no metadata match: 0 []
Metadata rows with no folder match: 0 []

Join verified: all 52 folders match exactly one metadata row, zero orphans either side.


,subject_id,n_wav_files,n_valid_indices,subject id,type,age,gender,education（years）,PHQ-9,CTQ-SF,LES,SSRS,GAD-7,PSQI
0,02010001,29,29,2010001,MDD,28,M,9,21,51,-2,18,10,5
1,02010002,29,29,2010002,MDD,18,F,12,23,77,-143,31,18,12
2,02010003,29,29,2010003,MDD,35,M,9,20,44,-100,28,16,15
3,02010004,29,29,2010004,MDD,25,F,19,12,53,-44,38,13,11
4,02010005,29,29,2010005,MDD,20,M,16,19,49,-3,28,11,5


In [7]:
print("Class balance (type):")
display(subjects_labeled_df["type"].value_counts())

print("\nGender x type crosstab:")
display(pd.crosstab(subjects_labeled_df["gender"], subjects_labeled_df["type"]))

print("\nAge by type:")
display(subjects_labeled_df.groupby("type")["age"].describe())

print("\nPHQ-9 by type:")
display(subjects_labeled_df.groupby("type")["PHQ-9"].describe())


Class balance (type):


type
HC     29
MDD    23
Name: count, dtype: int64


Gender x type crosstab:


type,HC,MDD
gender,,
F,9,7
M,20,16



Age by type:


,count,mean,std,min,25%,50%,75%,max
type,,,,,,,,
HC,29.0,31.551724,8.926630,19.0,23.0,31.0,38.0,52.0
MDD,23.0,30.913043,9.737057,18.0,23.0,28.0,38.5,52.0



PHQ-9 by type:


,count,mean,std,min,25%,50%,75%,max
type,,,,,,,,
HC,29.0,2.517241,2.114925,0.0,1.0,3.0,4.0,9.0
MDD,23.0,18.086957,4.440649,6.0,16.0,19.0,20.5,25.0


In [8]:
task_counts = (
    pd.Series([classify_task(i) for i in range(1, 30)])
    .value_counts()
    .rename_axis("task_type")
    .reset_index(name="n_file_indices")
)
display(task_counts)

# verify this mapping holds for every subject's actual file set, not just the 1-29 range in the abstract
per_subject_task_counts = []
for subject_dir in sorted(config.data.dataset_dir.iterdir()):
    if not subject_dir.is_dir():
        continue
    indices = [int(f.stem) for f in subject_dir.glob("*.wav")]
    per_subject_task_counts.append(pd.Series([classify_task(i) for i in indices]).value_counts())

task_counts_df = pd.DataFrame(per_subject_task_counts).fillna(0).astype(int)
uniform = (task_counts_df.nunique() == 1).all()
print(f"\nTask-type breakdown uniform across all 52 subjects: {bool(uniform.all())}")
display(task_counts_df.iloc[0])


,task_type,n_file_indices
0,interview,18
1,word_reading,6
2,picture_description,4
3,passage_reading,1



Task-type breakdown uniform across all 52 subjects: True


interview              18
word_reading            6
picture_description     4
passage_reading         1
Name: count, dtype: int64

In [9]:
print(
    f"{int(subjects_df['n_wav_files'].sum())} audio files map to {len(subjects_labeled_df)} unique subjects "
    f"(29 files/subject). Any train/test split MUST be done by subject_id, never by file — "
    f"splitting by file would put recordings from the same patient on both sides of the split."
)
assert subjects_labeled_df["subject_id"].is_unique, "subjects_labeled_df must have one row per subject"

save_processed(subjects_labeled_df, "subjects_labeled.csv")


1508 audio files map to 52 unique subjects (29 files/subject). Any train/test split MUST be done by subject_id, never by file — splitting by file would put recordings from the same patient on both sides of the split.
Saved 52 rows to /Users/vishalkarda/Documents/Projects/mendx.ai/backend/data/processed/subjects_labeled.csv


PosixPath('/Users/vishalkarda/Documents/Projects/mendx.ai/backend/data/processed/subjects_labeled.csv')

## Summary

- 52 subjects, 1508 recordings, uniform 29 files/subject (18 interview + 1 passage reading + 6 word reading + 4 picture description) — verified against every subject, not assumed.
- No unreadable/corrupt files and no exact-duplicate files detected (see the corrupt/duplicate-check cell above for the live count each run).
- Metadata workbook joins 1:1 against the on-disk subject folders after zero-padding subject IDs to 8 digits — zero orphans either side, hard-asserted above (not just printed).
- Class balance: MDD vs HC counts and demographics are in the cells above.
- Patient-level leakage: 29 recordings per subject means any downstream split must be by `subject_id`, never by file.
- The historical `config.py`/`data_loader.py` mismatch with this real layout (MDD/NC folder-split assumption, wrong metadata path) was fixed in `backend/dev/change_8_2026-08-16.md`; this notebook is the regression check for that fix.
- Output: `backend/data/processed/subjects_labeled.csv`, consumed by notebooks 02–04.
